# 01 — 下载 RQData 商品期货原始行情

本 Notebook 只负责：初始化 RQData、保存期货元数据、筛选商品品种、下载比例复权主力连续和可选的 99 指数连续。

本 Notebook 不计算收益率、波动率、相关性或市场状态。

In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cta_research  # noqa: E402
from cta_research import load_config  # noqa: E402

assert Path(cta_research.__file__).resolve().is_relative_to((PROJECT_ROOT / "src").resolve())

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
START_DATE = config["start_date"]
END_DATE = config["end_date"] or date.today().isoformat()
DOWNLOAD_INDEX_99 = bool(config["rqdata"]["download_index_99"])
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    name: PROJECT_ROOT / config["paths"][name]
    for name in ["future_instruments", "dominant_prices", "index_99_prices"]
}
cache_status = {name: paths[name].exists() for name in paths}
display(
    pd.Series(
        {
            **config,
            "DATA_DIR": str(DATA_DIR),
            "DOWNLOAD_INDEX_99": DOWNLOAD_INDEX_99,
            "RQDATA_END_DATE": END_DATE,
            "CACHE_STATUS": cache_status,
        },
        name="value",
    ).to_frame()
)
print("Existing caches will be incrementally updated.")

## 1. 初始化 RQData

In [ ]:
import rqdatac
from rqdatac import futures

rqdatac.init()
print("RQData initialized; credentials are not displayed or saved.")

## 2. 获取期货合约元数据

In [ ]:
future_instruments = rqdatac.all_instruments(type="Future")
if not isinstance(future_instruments, pd.DataFrame) or future_instruments.empty:
    raise ValueError("RQData returned no Future metadata.")
future_instruments.to_parquet(paths["future_instruments"], index=False)
print("Rows:", len(future_instruments))
print("Columns:", future_instruments.columns.tolist())
display(future_instruments.head())
display(future_instruments["exchange"].value_counts(dropna=False).rename("contracts").to_frame())

## 3. 筛选商品期货品种

In [ ]:
from cta_research import select_commodity_instruments

commodity_instruments, excluded_instruments = select_commodity_instruments(
    future_instruments,
    set(config["commodity_exchanges"]),
    config.get("excluded_symbols", {}),
)
commodity_symbols = sorted(commodity_instruments["underlying_symbol"].dropna().astype(str).str.upper().unique())
print("商品期货品种数量：", len(commodity_symbols))
print("商品期货品种：", commodity_symbols)
display(commodity_instruments.groupby("exchange")["underlying_symbol"].nunique().rename("symbols").to_frame())
display(excluded_instruments.groupby("exclusion_reason").size().rename("rows").to_frame())

# Ignore contracts whose entire lifetime precedes the requested research window.
symbol_end_dates = pd.Series(dtype="datetime64[ns]")
if "de_listed_date" in commodity_instruments:
    expiries = pd.to_datetime(commodity_instruments["de_listed_date"], errors="coerce")
    symbol_end_dates = expiries.groupby(commodity_instruments["underlying_symbol"]).max()
    commodity_symbols = [
        symbol for symbol in commodity_symbols
        if pd.isna(symbol_end_dates.get(symbol)) or symbol_end_dates[symbol] >= pd.Timestamp(START_DATE)
    ]

## 4. 下载比例复权主力连续日行情

In [ ]:
from cta_research import (
    group_incremental_requests,
    merge_rqdata_price_updates,
    normalize_rqdata_prices,
)

dominant_prices = pd.read_parquet(paths["dominant_prices"]) if paths["dominant_prices"].exists() else pd.DataFrame()
dominant_request_groups = group_incremental_requests(commodity_symbols, dominant_prices, START_DATE, END_DATE)
dominant_added = 0
for request_start, symbols in dominant_request_groups.items():
    for symbol in symbols:
        if pd.notna(symbol_end_dates.get(symbol)) and symbol_end_dates[symbol] < pd.Timestamp(request_start):
            continue
        rqdata_parameters = {
            "start_date": request_start,
            "end_date": END_DATE,
            "frequency": config["rqdata"]["frequency"],
            "fields": ["close"],
            "adjust_type": config["rqdata"]["adjust_type"],
            "adjust_method": config["rqdata"]["adjust_method"],
            "rule": config["rqdata"]["rule"],
            "rank": config["rqdata"]["rank"],
        }
        result = futures.get_dominant_price(symbol, **rqdata_parameters)
        if not isinstance(result, pd.DataFrame) or result.empty:
            raise RuntimeError(f"Empty dominant response: {symbol}, {request_start} — {END_DATE}; stopped.")
        updates = normalize_rqdata_prices(result)
        dominant_prices = merge_rqdata_price_updates(dominant_prices, updates)
        dominant_prices.to_parquet(paths["dominant_prices"], index=False)
        dominant_added += len(updates)
        print(f"Saved {symbol}: {len(updates)} rows")
print("主力连续行数：", len(dominant_prices))
print("主力连续新增行数：", dominant_added)
display(dominant_prices.head())

## 5. 可选下载 99 指数连续

In [ ]:
index_99_prices = pd.read_parquet(paths["index_99_prices"]) if paths["index_99_prices"].exists() else pd.DataFrame()
index_99_symbols = [f"{symbol}99" for symbol in commodity_symbols]
index_99_request_groups = group_incremental_requests(index_99_symbols, index_99_prices, START_DATE, END_DATE)
index_99_added = 0
if DOWNLOAD_INDEX_99:
    for request_start, symbols in index_99_request_groups.items():
        for order_book_id in symbols:
            symbol = order_book_id.removesuffix("99")
            if pd.notna(symbol_end_dates.get(symbol)) and symbol_end_dates[symbol] < pd.Timestamp(request_start):
                continue
            request_parameters = {
                "start_date": request_start,
                "end_date": END_DATE,
                "frequency": config["rqdata"]["frequency"],
                "fields": ["close"],
                "adjust_type": "none",
            }
            result = rqdatac.get_price(order_book_id, **request_parameters)
            if not isinstance(result, pd.DataFrame) or result.empty:
                raise RuntimeError(f"Empty index response: {order_book_id}, {request_start} — {END_DATE}; stopped.")
            updates = normalize_rqdata_prices(result, symbol_column="order_book_id")
            index_99_prices = merge_rqdata_price_updates(index_99_prices, updates)
            index_99_prices.to_parquet(paths["index_99_prices"], index=False)
            index_99_added += len(updates)
            print(f"Saved {order_book_id}: {len(updates)} rows")
    print("99指数连续新增行数：", index_99_added)
else:
    print("DOWNLOAD_INDEX_99=False; skipped.")

## 6. 下载完成

In [ ]:
print("RQData 原始行情已保存。")
print("主力连续：", paths["dominant_prices"])
if DOWNLOAD_INDEX_99:
    print("99 指数连续：", paths["index_99_prices"] if paths["index_99_prices"].exists() else "无返回数据")